# M10 — SMPL body through the static map (Parts A / A.5 / B)

Thin GPU runner for `tracking/smpl_person.py`. Logic lives in the fork; this notebook clones, installs, mounts Drive, sets paths, runs, displays.

- **Part A** — BEHAVE **GT** person (`person/fit02/person.ply`, no SMPL download) through the static surfel map.
- **Part A.5** — **MAMMA-estimated SMPL-X** (from the `run_mamma_behave` notebook) through the same map.
- **Part B** — root-trajectory forecast, ADE/FDE vs constant-velocity.

Pre-filled for **Date03_Sub05_boxtiny**; run on an A100 from the **gmail** account whose Drive holds the data. Outputs go straight to Drive. (boxtiny is in-place manipulation → Part B is a plumbing check; use a walking sequence for the real number.)

## 1. GPU check

In [ ]:
!nvidia-smi -L

## 2+3. Mount Drive + clone the fork + build CUDA submodules + deps
Adds `plyfile`/`trimesh` (read `person.ply`) and `smplx` (pose MAMMA params). The committed `render()` already accepts the means/rotation overrides `render_composite` uses — no runtime patch needed.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, sys
os.chdir('/content')
if not os.path.isdir('/content/gaussian-surfel-local-map'):
    !git clone --recursive https://github.com/steptang/gaussian-surfel-local-map.git
sys.path.insert(0, '/content/gaussian-surfel-local-map')
os.chdir('/content/gaussian-surfel-local-map')
!git pull -q   # need the branch with tracking/smpl_person.py
!pip install -q plyfile open3d scikit-learn scipy mediapy imageio imageio-ffmpeg trimesh opencv-python-headless smplx
!pip install -q --no-build-isolation ./submodules/diff-surfel-rasterization
!pip install -q --no-build-isolation ./submodules/simple-knn
print('ready')

## 4. Paths (pre-filled for boxtiny; outputs → Drive)
- `SMPL_ROOT` (Part A) — raw BEHAVE (`t*.000/person/fit02/person.ply`); same sequence as `CONV_ROOT`.
- `MAMMA_ROOT` / `MAMMA_MODEL_DIR` (Part A.5) — MAMMA exports + SMPL-X model dir on Drive (from `run_mamma_behave`). No `/content/mamma` here — separate session.
- Outputs `OUT_A` / `OUT_A5` write straight to Drive.

In [ ]:
DRIVE = '/content/drive/MyDrive/Research'
CONV_ROOT       = f'{DRIVE}/2dgs_output/datasets/behave_Date03_Sub05_boxtiny'
SMPL_ROOT       = f'{DRIVE}/datasets/behave/sequences/Date03_Sub05_boxtiny'   # Part A: raw BEHAVE person.ply
N_SELECT        = 24
MAMMA_ROOT      = f'{DRIVE}/datasets/mamma_exports/Date03_Sub05_boxtiny'        # Part A.5 exports
MAMMA_MODEL_DIR = f'{DRIVE}/2dgs_output/mamma_run/body_models/smplx_locked_head'
OUT_A  = f'{DRIVE}/2dgs_output/results/smpl_person/partA_gt'
OUT_A5 = f'{DRIVE}/2dgs_output/results/smpl_person/partA5_mamma'
import os
for d in (OUT_A, OUT_A5): os.makedirs(d, exist_ok=True)
assert os.path.isdir(CONV_ROOT), CONV_ROOT; assert os.path.isdir(SMPL_ROOT), SMPL_ROOT
print('paths OK')

## 5. Part A — GT person through the static map (+ Part B baseline)
Check the printed `timestep → frame` mapping, then eyeball `compare.png`. If the body floats/rotates/wrong-scale, add `--align_t x y z` / `--align_s`.

In [ ]:
!python -m tracking.smpl_person \
  --conv_root "$CONV_ROOT" --smpl_root "$SMPL_ROOT" \
  --smpl_source gt --view 0 --n_select $N_SELECT --out "$OUT_A" \
  --pred_hist 8 --pred_horizon 8

In [ ]:
import json
from IPython.display import Image, Video, display
OUT = OUT_A
print(json.dumps(json.load(open(f'{OUT}/metrics.json')), indent=2))
print(json.dumps(json.load(open(f'{OUT}/prediction.json')), indent=2))
display(Image(f'{OUT}/compare.png'))
display(Video(f'{OUT}/sequence.mp4', embed=True, width=720))

## 6. Part A.5 — MAMMA-estimated SMPL-X through the static map
Needs the MAMMA exports + SMPL-X model dir on Drive (from **`M10 - run_mamma_behave`** §7). Since MAMMA used your BEHAVE calibration, its body is already in the surfel-map world frame — if it lands on the person, the calibration convention was right; if not, re-run MAMMA §5 with `--extrinsics cam2world` / `--quat_order xyzw`.

In [ ]:
!python -m tracking.smpl_person \
  --conv_root "$CONV_ROOT" --smpl_root "$MAMMA_ROOT" --smpl_model_dir "$MAMMA_MODEL_DIR" \
  --smpl_source mamma --view 0 --n_select $N_SELECT --out "$OUT_A5" \
  --pred_hist 8 --pred_horizon 8

In [ ]:
import json
from IPython.display import Image, Video, display
OUT = OUT_A5
print(json.dumps(json.load(open(f'{OUT}/metrics.json')), indent=2))
print(json.dumps(json.load(open(f'{OUT}/prediction.json')), indent=2))
display(Image(f'{OUT}/compare.png'))
display(Video(f'{OUT}/sequence.mp4', embed=True, width=720))

## 7. Compare GT (A) vs MAMMA (A.5)

In [ ]:
import json
for tag, OUT in [('GT (A)', OUT_A), ('MAMMA (A.5)', OUT_A5)]:
    try:
        m = json.load(open(f'{OUT}/metrics.json')); p = json.load(open(f'{OUT}/prediction.json'))
        print(f"{tag:12s} person_psnr_mean={m['person_psnr_mean']:.2f}  "
              f"const_vel_ADE={p['const_vel_ADE']:.4f}  FDE={p['const_vel_FDE']:.4f}")
    except FileNotFoundError:
        print(f"{tag:12s} not run yet ({OUT})")